# 12. 연도별 패널데이터 전처리

## 목적
2020~2025년 **어린이 교통사고 + 0~12세 어린이 인구 + 자동차 등록대수**를 229개 시군구 × 연도 기준으로 통합한다.

최종 산출물
- `data_clean/population_2020_2025_child_clean.csv`
- `data_clean/vehicle_2020_2025_clean.csv`
- `data_processed/panel_2020_2025_229.csv`

기존 사고 데이터의 229개 시군구를 기준 지역 목록으로 사용한다.


In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

BASE = Path("..")
RAW = BASE / "data_raw"
CLEAN = BASE / "data_clean"
PROCESSED = BASE / "data_processed"

def normalize_region(x):
    x = re.sub(r"\s+", " ", str(x).strip())
    x = x.replace("강원도 ", "강원특별자치도 ")
    x = x.replace("전라북도 ", "전북특별자치도 ")
    x = x.replace("제주도 ", "제주특별자치도 ")
    x = x.replace("경상북도 군위군", "대구광역시 군위군")
    if x == "세종특별자치시":
        x = "세종특별자치시 세종시"
    return x


### 1. 기준 229개 지역 만들기
사고 데이터는 이미 2020~2025 × 229개 시군구로 정리되어 있으므로 이를 기준으로 삼는다.


In [ ]:
acc = pd.read_csv(CLEAN / "accident_2020_2025_child_clean.csv")
acc["sido_sgg_nm"] = acc["sido_sgg_nm"].map(normalize_region)
master_regions = sorted(acc["sido_sgg_nm"].unique())
print("기준 지역 수:", len(master_regions))
assert len(master_regions) == 229


### 2. 2020~2025 연도별 0~12세 어린이 인구 정리
행정안전부 연령별 인구현황 원본의 `연령구간인구수`가 0~12세 합계다.


In [ ]:
population_parts = []
for year in range(2020, 2026):
    path = RAW / f"{year}12_{year}12_연령별인구현황_연간.csv"
    temp = pd.read_csv(path, encoding="cp949")
    temp["sido_sgg_nm"] = (
        temp["행정구역"]
        .str.replace(r"\s*\(\d+\)\s*$", "", regex=True)
        .str.strip()
        .map(normalize_region)
    )
    value_col = f"{year}년_계_연령구간인구수"
    temp[value_col] = (
        temp[value_col].astype(str).str.replace(",", "", regex=False).astype(int)
    )
    temp = temp[temp["sido_sgg_nm"].isin(master_regions)][["sido_sgg_nm", value_col]].copy()
    temp["year"] = year
    temp = temp.rename(columns={value_col: "child_population_0_12"})
    temp = temp.drop_duplicates(["year", "sido_sgg_nm"], keep="last")
    population_parts.append(temp)

population = pd.concat(population_parts, ignore_index=True)
population = population[["year", "sido_sgg_nm", "child_population_0_12"]]
print(population.shape)
print(population.groupby("year")["sido_sgg_nm"].nunique())
population.to_csv(CLEAN / "population_2020_2025_child_clean.csv", index=False, encoding="utf-8-sig")


### 3. 자동차 등록대수 정리
KOSIS 원본 `.xls`는 실제로 SpreadsheetML(XML) 형식이다. 복합시(수원시, 성남시 등)는 자동차 자료가 하위 구 단위로 제공되는 경우가 있어, **기준 229개 지역의 부모 시 단위로 합산**한다.


In [ ]:
vehicle_path = RAW / "116_DT_MLTM_5498_20261006015915(1).xls"
text = vehicle_path.read_bytes().decode("euc-kr", errors="replace")

row_blocks = re.findall(r"<Row[^>]*>(.*?)</Row>", text, flags=re.S)
records = []
for rb in row_blocks:
    vals = []
    for cb in re.findall(r"<Cell[^>]*>(.*?)</Cell>", rb, flags=re.S):
        m = re.search(r"<Data[^>]*>(.*?)</Data>", cb, flags=re.S)
        vals.append(re.sub(r"<[^>]+>", "", m.group(1)).strip() if m else "")
    if vals:
        records.append(vals)

header_idx = next(i for i, r in enumerate(records) if len(r) >= 5 and r[0] == "시도명" and r[1] == "시군구")
header = records[header_idx]
year_cols = {int(re.match(r"(\d{4})", h).group(1)): i for i, h in enumerate(header) if re.match(r"\d{4}\.12", h)}

sido_map = {
    "서울":"서울특별시", "부산":"부산광역시", "대구":"대구광역시", "인천":"인천광역시",
    "광주":"광주광역시", "대전":"대전광역시", "울산":"울산광역시", "세종":"세종특별자치시",
    "경기":"경기도", "강원":"강원특별자치도", "충북":"충청북도", "충남":"충청남도",
    "전북":"전북특별자치도", "전남":"전라남도", "경북":"경상북도", "경남":"경상남도", "제주":"제주특별자치도"
}

direct, child_sum = {}, {}
for r in records[header_idx + 1:]:
    if len(r) < 5:
        continue
    sido, sgg, level, item = r[0].strip(), r[1].strip(), r[2].strip(), r[3].strip()
    if level != "총계" or item != "계" or sgg == "계":
        continue
    full_sido = sido_map.get(sido, sido)
    key = "세종특별자치시 세종시" if full_sido == "세종특별자치시" else normalize_region(f"{full_sido} {sgg}")
    mode = None
    if key in master_regions:
        parent = key; mode = "direct"
    else:
        parts = sgg.split()
        if len(parts) >= 2:
            candidate = normalize_region(f"{full_sido} {parts[0]}")
            if candidate in master_regions and parts[0].endswith("시"):
                parent = candidate; mode = "child"
    if mode is None:
        continue
    for year, idx in year_cols.items():
        if year not in range(2020, 2026) or idx >= len(r):
            continue
        value = int(float(str(r[idx]).replace(",", "")))
        if mode == "direct":
            direct[(year, parent)] = value
        else:
            child_sum[(year, parent)] = child_sum.get((year, parent), 0) + value

vehicle_rows = []
for year in range(2020, 2026):
    for region in master_regions:
        value = direct.get((year, region), child_sum.get((year, region)))
        vehicle_rows.append([year, region, value])
vehicle = pd.DataFrame(vehicle_rows, columns=["year", "sido_sgg_nm", "vehicle_count"])
print(vehicle.shape)
print(vehicle.isna().sum())
vehicle.to_csv(CLEAN / "vehicle_2020_2025_clean.csv", index=False, encoding="utf-8-sig")


### 4. 사고 + 인구 + 자동차 병합
연도와 시군구를 동시에 키로 사용한다.


In [ ]:
acc_panel = acc.rename(columns={
    "std_year":"year", "acc_cnt":"accident_count",
    "dth_dnv_cnt":"death_count", "injpsn_cnt":"injury_count"
})[["year", "sido_sgg_nm", "accident_count", "death_count", "injury_count"]]

panel = (
    acc_panel
    .merge(population, on=["year", "sido_sgg_nm"], how="left")
    .merge(vehicle, on=["year", "sido_sgg_nm"], how="left")
)
panel["accident_per_10k_child"] = panel["accident_count"] / panel["child_population_0_12"] * 10000
panel["vehicles_per_100_child"] = panel["vehicle_count"] / panel["child_population_0_12"] * 100

panel = panel[[
    "year", "sido_sgg_nm", "child_population_0_12", "vehicle_count",
    "accident_count", "death_count", "injury_count",
    "accident_per_10k_child", "vehicles_per_100_child"
]].sort_values(["year", "sido_sgg_nm"])

print("패널 크기:", panel.shape)
print("지역 수:", panel["sido_sgg_nm"].nunique())
print("결측치:", panel.isna().sum().sum())
assert panel.shape == (229 * 6, 9)
assert panel.isna().sum().sum() == 0
panel.to_csv(PROCESSED / "panel_2020_2025_229.csv", index=False, encoding="utf-8-sig")
panel.head()


완성된 패널은 **1,374행 = 229개 시군구 × 6개 연도**이며, 새 AI 모델의 기반 데이터로 사용한다.
